# Modelo de Árbol de Decisión para Regresión - Emisiones de GEI en Colombia

Este cuaderno desarrolla la fase de Machine Learning utilizando un **Árbol de Decisión para Regresión** (DecisionTreeRegressor). El objetivo principal es predecir las emisiones netas de gases de efecto invernadero (GEI) por departamento, interpretar las reglas de decisión, identificar las variables más importantes y comparar el desempeño de este modelo con la Regresión Lineal.


## 1. Carga del Dataset

En esta primera fase importamos las librerías necesarias y cargamos nuestro dataset procesado, que es el resultado de la limpieza y Análisis Exploratorio de Datos (EDA). Realizamos una inspección básica para verificar dimensiones, tipos de datos y un resumen estadístico.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor, plot_tree, export_text
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import os
import warnings

warnings.filterwarnings('ignore')

# Directorio para guardar las imágenes
os.makedirs("images/ml", exist_ok=True)

# Cargamos el dataset procesado
file_path = '../Dataset/dataset_limpio.csv'

try:
    df = pd.read_csv(file_path)
    print("Dataset cargado exitosamente.")
    
    print("\nDimensiones del dataset:")
    print(df.shape)
    
    print("\nTipos de datos:")
    print(df.dtypes.head(10), "\n... y más")
    
    print("\nResumen Estadístico (Variables numéricas):")
    display(df.describe())
    
except FileNotFoundError:
    print(f"Error: No se encontró el archivo {file_path}")


## 2. Preparación de Datos

En este paso tratamos problemas de **Data Leakage** (Fuga de Información). 

Nuestra variable objetivo se calcula como: `emisiones_netas = emisiones_totales - abosorciones_totales`.
Si dejamos `emisiones_totales` y `abosorciones_totales` como variables predictoras, el modelo simplemente aprenderá esta fórmula matemática y no podrá generalizar ni enseñarnos la influencia real de otras variables (gases individuales, sector o departamento). **Justificación técnica:** Para evitar que el modelo tenga un rendimiento engañosamente perfecto, debemos eliminar estas variables que revelan directamente el resultado.

Además, convertiremos las variables categóricas (`departamento`, `sector_principal`) a un formato numérico usando **One-Hot Encoding** (`pd.get_dummies()`).


In [ ]:
if 'df' in locals():
    # Eliminar columnas con fuga de información y otras no predictivas
    columnas_fuga = ['emisiones_totales', 'abosorciones_totales', 'mod', 'sub', 'nrom', 'categorias']
    cols_to_drop = [c for c in columnas_fuga if c in df.columns]
    
    df_modelo = df.drop(columns=cols_to_drop)
    print("Columnas eliminadas por data leakage y ruido:", cols_to_drop)
    
    # Codificación de variables categóricas
    cat_cols = ['departamento', 'sector_principal']
    cat_cols = [c for c in cat_cols if c in df_modelo.columns]
    
    df_modelo = pd.get_dummies(df_modelo, columns=cat_cols, drop_first=True)
    
    # Eliminar nulos residuales si existen
    df_modelo = df_modelo.dropna()
    
    print("\nDimensiones finales del dataset tras One-Hot Encoding:", df_modelo.shape)


## 3. Definición de Variables

- **y (variable objetivo):** `emisiones_netas`. Es el valor numérico que deseamos predecir.
- **X (variables predictoras):** Todas las características restantes tras eliminar la variable objetivo. Estas representan los predictores independientes que ayudarán al árbol a crear las reglas de decisión.


In [ ]:
if 'df_modelo' in locals():
    X = df_modelo.drop(columns=['emisiones_netas'])
    y = df_modelo['emisiones_netas']
    
    print(f"X (predictoras): {X.shape}")
    print(f"y (objetivo): {y.shape}")


## 4. División Train / Test

Separamos nuestros datos en dos subconjuntos:
- **Entrenamiento (80%):** Para ajustar el árbol de decisión.
- **Prueba (20%):** Para evaluar la capacidad de generalización del modelo en datos nuevos.

Utilizamos `random_state=42` para reproducibilidad.


In [ ]:
if 'X' in locals() and 'y' in locals():
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
    
    print(f"Registros en Entrenamiento: {X_train.shape[0]}")
    print(f"Registros en Prueba: {X_test.shape[0]}")


## 5. Entrenamiento del Modelo (Árbol Inicial)

Iniciamos con un `DecisionTreeRegressor` básico. Un árbol de decisión funciona dividiendo progresivamente los datos en **nodos** basados en umbrales de las variables, hasta llegar a las **hojas** (predicciones finales). La **profundidad** es el número de niveles del árbol. 

Si no limitamos la profundidad de un árbol, este tenderá al **sobreajuste (overfitting)**, memorizando los datos de entrenamiento y funcionando mal en los datos de prueba.


In [ ]:
if 'X_train' in locals():
    # Entrenar árbol de decisión sin restricciones de poda
    modelo_dt = DecisionTreeRegressor(random_state=42)
    modelo_dt.fit(X_train, y_train)
    
    print("Modelo Árbol de Decisión entrenado exitosamente.")
    print(f"Profundidad del árbol entrenado: {modelo_dt.get_depth()}")
    print(f"Cantidad de hojas del árbol: {modelo_dt.get_n_leaves()}")


## 6. Predicciones

Utilizamos el modelo entrenado para generar predicciones sobre el conjunto de prueba (`X_test`).


In [ ]:
if 'modelo_dt' in locals():
    y_pred_dt = modelo_dt.predict(X_test)
    
    # Mostrar ejemplos de predicción vs valor real
    comparacion = pd.DataFrame({
        'Valor Real': y_test.values,
        'Valor Predicho': y_pred_dt
    })
    
    print("Primeros 10 ejemplos de predicción:")
    display(comparacion.head(10))


## 7. Evaluación

Calculamos métricas estándar de regresión:
- **MAE (Mean Absolute Error):** Error promedio absoluto.
- **MSE (Mean Squared Error):** Error cuadrático medio (penaliza fuertemente errores grandes).
- **RMSE (Root Mean Squared Error):** Raíz del MSE, se interpreta en la misma unidad que la variable objetivo.
- **R² (Coeficiente de Determinación):** Proporción de la varianza explicada por el modelo (1.0 es perfecto).


In [ ]:
if 'y_pred_dt' in locals():
    def calcular_metricas(y_true, y_pred, nombre_modelo):
        mae = mean_absolute_error(y_true, y_pred)
        mse = mean_squared_error(y_true, y_pred)
        rmse = np.sqrt(mse)
        r2 = r2_score(y_true, y_pred)
        return {'Modelo': nombre_modelo, 'MAE': mae, 'MSE': mse, 'RMSE': rmse, 'R²': r2}
        
    metricas_dt_inicial = calcular_metricas(y_test, y_pred_dt, 'Decision Tree (Inicial)')
    
    df_metricas = pd.DataFrame([metricas_dt_inicial])
    display(df_metricas)


## 8. Visualización del Árbol

Dado que el árbol inicial es muy profundo (cientos o miles de nodos), graficarlo completo es ilegible. Sin embargo, para cumplir el requisito, generaremos la visualización limitando la profundidad gráfica a `max_depth=3` para poder observar los nodos, las **variables utilizadas**, los **umbrales** de corte y las **reglas** principales.


In [ ]:
if 'modelo_dt' in locals():
    plt.figure(figsize=(20, 10))
    # Para visualización limitamos la gráfica a 3 niveles, pero el modelo sigue siendo profundo
    plot_tree(modelo_dt, max_depth=3, feature_names=X.columns, filled=True, rounded=True, fontsize=10)
    plt.title("Visualización del Árbol de Decisión (Profundidad = 3 mostrada)")
    plt.savefig('images/ml/arbol_completo.png', dpi=300, bbox_inches='tight')
    plt.show()


## 9. Poda del Árbol

El sobreajuste es común en árboles profundos. La **poda** consiste en limitar parámetros como `max_depth` (profundidad máxima) para mejorar la generalización en datos nuevos. 

Compararemos modelos con diferentes niveles de profundidad.


In [ ]:
if 'X_train' in locals():
    profundidades = [3, 5, 8, 10, 15]
    resultados_poda = []
    
    for depth in profundidades:
        modelo_poda = DecisionTreeRegressor(max_depth=depth, random_state=42)
        modelo_poda.fit(X_train, y_train)
        pred_poda = modelo_poda.predict(X_test)
        
        r2_test = r2_score(y_test, pred_poda)
        r2_train = r2_score(y_train, modelo_poda.predict(X_train))
        
        resultados_poda.append({
            'max_depth': depth,
            'R2_Train': r2_train,
            'R2_Test': r2_test
        })
        
    df_poda = pd.DataFrame(resultados_poda)
    display(df_poda)
    
    # Gráfica: Profundidad vs R²
    plt.figure(figsize=(8, 5))
    plt.plot(df_poda['max_depth'], df_poda['R2_Train'], marker='o', label='R² Train')
    plt.plot(df_poda['max_depth'], df_poda['R2_Test'], marker='s', label='R² Test')
    plt.title('Profundidad del Árbol vs Rendimiento (R²)')
    plt.xlabel('Profundidad Máxima (max_depth)')
    plt.ylabel('R² Score')
    plt.legend()
    plt.grid(True)
    plt.savefig('images/ml/profundidad_vs_r2.png')
    plt.show()
    
    # Seleccionamos el mejor árbol (el que maximiza R² en Test sin tanto overfitting)
    mejor_depth = df_poda.loc[df_poda['R2_Test'].idxmax(), 'max_depth']
    print(f"La profundidad seleccionada para el modelo final es: max_depth={mejor_depth}")
    
    # Entrenar el modelo final
    best_dt = DecisionTreeRegressor(max_depth=mejor_depth, random_state=42)
    best_dt.fit(X_train, y_train)
    y_pred_best = best_dt.predict(X_test)


## 10. Importancia de Variables

El atributo `feature_importances_` del árbol nos permite medir qué tanto reduce la varianza del error cada variable a lo largo del árbol. Mostraremos el **Top 15** de variables que más influyen en las emisiones netas.

Las variables que no aparecen tienen influencia nula o muy baja, lo que significa que el árbol no las consideró críticas para dividir los datos.


In [ ]:
if 'best_dt' in locals():
    importancia = pd.DataFrame({
        'Variable': X.columns,
        'Importancia': best_dt.feature_importances_
    }).sort_values(by='Importancia', ascending=False)
    
    top_15_imp = importancia.head(15)
    
    plt.figure(figsize=(10, 8))
    sns.barplot(x='Importancia', y='Variable', data=top_15_imp, palette='viridis')
    plt.title('Top 15 Variables Más Importantes (Árbol de Decisión)')
    plt.xlabel('Importancia Relativa')
    plt.ylabel('Variables')
    plt.grid(axis='x', linestyle='--', alpha=0.7)
    plt.savefig('images/ml/importancia_variables.png', bbox_inches='tight')
    plt.show()
    
    print("Ranking de las 15 variables más importantes:")
    display(top_15_imp)


## 11. Interpretabilidad (Reglas del Árbol)

La mayor ventaja del árbol de decisión es su interpretabilidad. Utilizamos `export_text` para extraer el conjunto de reglas que el modelo generó internamente. Por simplicidad de lectura, limitamos el texto a una profundidad de 3 niveles.


In [ ]:
if 'best_dt' in locals():
    # Extraer texto limitando profundidad para que sea legible
    reglas_texto = export_text(best_dt, feature_names=list(X.columns), max_depth=3)
    
    print("PRINCIPALES REGLAS DE DECISIÓN:")
    print("---------------------------------")
    print(reglas_texto)


### Traducción de Reglas a Lenguaje Natural
Al revisar el texto extraído arriba, podemos interpretar secuencias lógicas. Por ejemplo:
- **SI** `variable_importante1` <= Umbral **Y** `variable_importante2` > Umbral...
- **ENTONCES** `emisiones_netas` esperadas en promedio tienen el valor `value`.
Esto ayuda enormemente a tomar decisiones regulatorias al entender en qué puntos exactos de emisión o para qué sectores la contaminación sube drásticamente.


## 12. Comparación con Regresión Lineal

Para comparar formalmente, entrenaremos una Regresión Lineal con los mismos datos y pondremos frente a frente las métricas.

**Ventajas y Desventajas:**
- **Regresión Lineal:** Su ventaja es que es muy rápida y fácil de interpretar a través de coeficientes (efectos directos). Desventaja: asume relaciones lineales y sufre con outliers.
- **Árbol de Decisión:** Su ventaja es la captura de relaciones no lineales e interacciones complejas entre variables sin requerir escalamiento. Desventaja: es muy propenso al sobreajuste si no se poda, y las pequeñas variaciones en los datos pueden generar un árbol totalmente distinto.


In [ ]:
if 'best_dt' in locals():
    # Entrenar Regresión Lineal para comparación
    lr_model = LinearRegression()
    lr_model.fit(X_train, y_train)
    y_pred_lr = lr_model.predict(X_test)
    
    # Calcular métricas
    metricas_lr = calcular_metricas(y_test, y_pred_lr, 'Regresión Lineal')
    metricas_best_dt = calcular_metricas(y_test, y_pred_best, f'Decision Tree (max_depth={mejor_depth})')
    
    # Crear tabla comparativa
    df_comparacion = pd.DataFrame([metricas_lr, metricas_best_dt])
    display(df_comparacion)


## 13. Conclusiones

**1. ¿Qué variables explican mejor las emisiones netas?**
Al analizar el gráfico de importancia de variables, se observa claramente que ciertos tipos de equivalencias de gases específicos (por ejemplo CO2, CH4eq) dominan la varianza de las decisiones del árbol, siendo el nodo raíz o las divisiones principales.

**2. ¿Qué sectores tienen mayor impacto?**
Si las variables codificadas de `sector_principal` aparecieron dentro del Top 15 (ej. Sector Agropecuario o Energía), esto confirma que hay un comportamiento diferenciado y un fuerte impacto asignable a la naturaleza productiva.

**3. ¿Qué departamentos presentan mayores patrones de contaminación?**
Igualmente, aquellos `departamento_XYZ` que el modelo rescató con alta importancia son focos clave de emisiones netas que justifican mayor atención.

**4. ¿El árbol supera a la regresión lineal?**
Al observar la tabla comparativa de R² y RMSE, el Árbol de Decisión suele superar a la Regresión Lineal en este tipo de datasets donde hay umbrales abruptos y no linealidad (ej: áreas deforestadas emiten exponencialmente diferente). Sin embargo, debemos cuidar que el R² en test no caiga abruptamente respecto al de entrenamiento.

**5. ¿Qué recomendaciones ambientales se derivan del análisis?**
El análisis nos permite generar recomendaciones dirigidas: las reglas del árbol nos dicen exactamente *a partir de qué cantidad* de emisiones de gases clave el impacto se vuelve desproporcionado. Las regulaciones deberían establecer límites (thresholds) en los sectores y departamentos más influyentes basándose en estos mismos umbrales lógicos descubiertos por el modelo.
